### 1. Khai báo thư viện (Imports)
- 🎯 **Mục đích:** Tải các công cụ cần thiết vào bộ nhớ.
- 💻 **Giải thích Code:** Sử dụng `pandas` để xử lý bảng, `matplotlib/seaborn` vẽ biểu đồ, `scikit-learn` chuẩn hoá và chia data, `torch` xây dựng mạng MLP.
- 📊 **Áp dụng vào Dữ liệu:** Thiết lập nền tảng môi trường cho toàn bộ bài toán phân tích giá nhà Boston.
- 📈 **Kết quả (Output):** Môi trường sẵn sàng hoạt động.


In [19]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Cài đặt phong cách biểu đồ chung
sns.set_theme(style="whitegrid")

### 2. Đọc Dữ liệu & EDA Tổng quan (Load Dataset & Overview EDA)
- 🎯 **Mục đích:** Tải dữ liệu từ ổ cứng và làm sạch sơ bộ định dạng.
- 💻 **Giải thích Code:** `pd.read_csv` nạp file CSV, tham số `na_values` nhận diện các khoảng trắng thành giá trị rỗng chuẩn `NaN`. Dùng thêm `replace` regex để dọn dẹp triệt để.
- 📊 **Áp dụng vào Dữ liệu:** Quét qua 506 căn nhà với 14 đặc trưng.
- 📈 **Kết quả (Output):** In ra cấu trúc 5 dòng đầu và thông tin DataFrame.

**Chú thích ý nghĩa các cột (Features):**
- **CRIM:** Tỉ lệ tội phạm bình quân đầu người theo thị trấn.
- **ZN:** Tỉ lệ quy hoạch đất thổ cư (diện tích các lô lớn hơn 25.000 sq.ft).
- **INDUS:** Tỉ lệ diện tích đất kinh doanh (phi bán lẻ) trên mỗi thị trấn.
- **CHAS:** Biến giả sông Charles (bằng 1 nếu đất bao quanh sông; 0 nếu ngược lại).
- **NOX:** Nồng độ khí Nitơ Oxit (phần 10 triệu) - Thể hiện mức độ ô nhiễm không khí.
- **RM:** Số lượng phòng ở trung bình trên mỗi căn hộ.
- **AGE:** Tỉ lệ các căn nhà (có chủ sở hữu) được xây dựng trước năm 1940.
- **DIS:** Khoảng cách (có trọng số) đến 5 trung tâm việc làm lớn ở Boston.
- **RAD:** Chỉ số đánh giá khả năng tiếp cận/đi lại đến các đường cao tốc.
- **TAX:** Thuế suất tài sản tính trên 10.000 USD.
- **PTRATIO:** Tỉ lệ học sinh / giáo viên theo từng thị trấn.
- **B:** Chỉ số toán học đại diện cho tỉ lệ người da đen tại khu vực.
- **LSTAT:** Tỉ lệ phần trăm dân số có mức thu nhập/trạng thái xã hội thấp.
- **MEDV (Biến Mục Tiêu - Target):** Giá trị trung bình của những ngôi nhà được chủ sở hữu cư trú (Đơn vị: nghìn USD).


In [ ]:
df = pd.read_csv("../data/raw/Boston-house-price-data.csv", na_values=["", " ", "nan", "NaN", "null", "NULL"])
df = df.replace(r"^\s*$", np.nan, regex=True)

print("Kích thước dữ liệu gốc:", df.shape)
display(df.head())
df.info()

print("\n--- THỐNG KÊ TỔNG QUAN (DESCRIBE) ---")
display(df.describe())


### 3. Định nghĩa Biến số & Phân chia Dữ liệu (Variable Definition & Data Splitting)
- 🎯 **Mục đích:** Tách loại biến số và phân chia dữ liệu ngay sau EDA tổng quan để **ngăn chặn rò rỉ dữ liệu (Data Leakage)**.
- 💻 **Giải thích Code:** Xác định các cột số (`numeric_cols`) và cột features (`feature_cols`). Sau đó chia thành 3 tập (70% Train / 15% Val / 15% Test).
- ⚠️ **Nguyên tắc:** Phân chia dữ liệu TRƯỚC khi phân tích chuyên sâu (Missing, Duplicates, Outliers). Toàn bộ EDA sâu và tiền xử lý chỉ dựa trên tập **Train**.


In [ ]:
# Khai báo các cột số và cột features (loại bỏ MEDV)
numeric_cols = df.select_dtypes(include='number').columns
feature_cols = [col for col in numeric_cols if col != 'MEDV']
print(f"Số cột số: {len(numeric_cols)}, Feature columns: {list(feature_cols)}")

# Phân chia Train/Val/Test (70/15/15)
X = df[feature_cols].copy()
y = df['MEDV'].copy()

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

# Tạo DataFrame đầy đủ để phân tích EDA
df_train = X_train.copy(); df_train['MEDV'] = y_train
df_val   = X_val.copy();   df_val['MEDV']   = y_val
df_test  = X_test.copy();  df_test['MEDV']  = y_test

print(f"\nTập Train: {df_train.shape}")
print(f"Tập Val:   {df_val.shape}")
print(f"Tập Test:  {df_test.shape}")


### 4. Kiểm tra Dữ liệu Thiếu (Missing) và Trùng lặp (Duplicates) — CHỈ TRÊN TẬP TRAIN
- 🎯 **Mục đích:** Khảo sát độ sạch của **tập Train** trước khi huấn luyện.
- 💻 **Giải thích Code:** `df.isna().sum()` đếm dữ liệu rỗng. `df.duplicated().sum()` tìm dòng giống nhau. Vẽ Heatmap và Pie chart để dễ nhìn.
- 📊 **Áp dụng vào Dữ liệu:** Phát hiện cột nào thiếu nhiều nhất (thường là ZN, CHAS...).
- 📈 **Kết quả (Output):** Các thông số báo lỗi và 2 biểu đồ trực quan.


In [ ]:
missing_counts = df_train.isna().sum()
dup_count = df_train.duplicated().sum()

print(f"Tổng số ô bị thiếu (Missing): {missing_counts.sum()}")
print(f"Tổng số dòng bị trùng lặp (Duplicated): {dup_count}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap Missing
sns.heatmap(missing_counts.to_frame().T, annot=True, fmt='d', cmap='YlOrRd', yticklabels=False, cbar=True, ax=axes[0])
axes[0].set_title('Heatmap Missing Values (Train)', fontsize=14, fontweight='bold')

# Pie chart Duplicated
labels = ['Unique', 'Duplicated']
sizes = [len(df_train) - dup_count, dup_count]
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', colors=['#3498db', '#e74c3c'], startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Duplicated Rows Distribution (Train)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


### 5. Phân tích Ngoại lai (Outliers Analysis) — CHỈ TRÊN TẬP TRAIN
- 🎯 **Mục đích:** Khảo sát các điểm dữ liệu dị biệt (Outliers) bằng Boxplot để quyết định phương pháp xử lý phù hợp.
- 💻 **Giải thích Code:** Tính toán `Q1, Q3, Bounds` cho từng biến và vẽ `Boxplot`. Các đường đứt nét màu đỏ thể hiện giới hạn râu (whisker bounds). Các điểm nằm ngoài đường này là Outliers.
- 📊 **Áp dụng vào Dữ liệu:** Nhận thấy các biến `CRIM`, `ZN`, `B`, `LSTAT` có rất nhiều outliers tự nhiên do đặc thù của giá nhà.
- 📈 **Kết quả (Output):** Biểu đồ Boxplot cho phép phát hiện mức độ phân tán.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

# Tính IQR và vẽ Boxplot để quan sát
for i, col in enumerate(numeric_cols):
    # Tính Bounds
    Q1 = df_train[col].quantile(0.25)
    Q3 = df_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Vẽ Boxplot
    sns.boxplot(y=df_train[col], ax=axes[i], color='#3498db')
    axes[i].axhline(lower_bound, color='red', linestyle='--', alpha=0.5)
    axes[i].axhline(upper_bound, color='red', linestyle='--', alpha=0.5)
    axes[i].set_title(col, fontweight='bold')

# Ẩn các ô trống thừa
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outliers Analysis Boxplot (Red dashed = Bounds)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Đã hoàn tất quan sát phân bố Outliers.")

### 6. Tiền xử lý cả 3 tập: Xóa Missing & Duplicates (Dựa trên kết quả EDA từ tập Train)
- 🎯 **Mục đích:** Áp dụng các quyết định xử lý (rút ra từ phân tích tập Train) cho cả 3 tập Train, Val, Test.
- 💻 **Giải thích Code:** `dropna()` xóa các dòng có giá trị thiếu. `drop_duplicates()` xóa các dòng trùng lặp.
- ⚠️ **Nguyên tắc:** Quyết định xử lý được xác định từ tập Train, nhưng áp dụng đồng nhất cho cả 3 tập.


In [ ]:
# Xóa Missing Values trên cả 3 tập
print("--- TRƯỚC KHI XÓA MISSING ---")
print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")

df_train = df_train.dropna().reset_index(drop=True)
df_val   = df_val.dropna().reset_index(drop=True)
df_test  = df_test.dropna().reset_index(drop=True)

# Xóa Duplicates trên cả 3 tập
df_train = df_train.drop_duplicates().reset_index(drop=True)
df_val   = df_val.drop_duplicates().reset_index(drop=True)
df_test  = df_test.drop_duplicates().reset_index(drop=True)

print("\n--- SAU KHI XÓA MISSING & DUPLICATES ---")
print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")


### 6.1. Xử lý Outliers bằng Biến đổi Logarit (Log Transformation) — Detect từ Train, Apply cả 3 tập
- 🎯 **Mục đích:** Xử lý Outliers tự nhiên mà không làm mất thông tin bằng cách biến đổi Logarit.
- 💻 **Giải thích Code:** Sử dụng `np.log1p()` (tính log(1+x)) để co cụm khoảng cách của các dữ liệu lớn, giúp đường cong phân phối trở nên mượt mà và gần với phân phối chuẩn (Normal Distribution) hơn.
- 📊 **Áp dụng vào Dữ liệu:** Áp dụng cho `CRIM`, `ZN`, `B`, `LSTAT` là các biến lệch phải/trái nặng.
- 📈 **Kết quả (Output):** Biểu đồ KDE cho thấy sự thay đổi từ sắc nhọn sang hình chuông mượt mà.

In [ ]:
# Tự động phát hiện các cột bị lệch nặng từ tập Train
SKEW_THRESHOLD = 1.0

skew_info = {}
for col in feature_cols:
    skewness = df_train[col].skew()
    if abs(skewness) > SKEW_THRESHOLD:
        if skewness < -SKEW_THRESHOLD:
            # Lệch trái cực đoan -> Reflect & Log
            transform_type = 'reflect_log'
        else:
            # Lệch phải -> Log
            transform_type = 'log'
        skew_info[col] = {'skewness': round(skewness, 4), 'transform': transform_type}
        if transform_type == 'reflect_log':
            skew_info[col]['max_val'] = df_train[col].max()  # Lấy max từ TRAIN

print("Các cột cần biến đổi (phát hiện từ tập Train):")
for col, info in skew_info.items():
    print(f"  - {col}: skewness={info['skewness']}, transform={info['transform']}")

# Visualization trước/sau trên tập Train
cols_to_transform = list(skew_info.keys())
n_cols = len(cols_to_transform)

if n_cols > 0:
    fig, axes = plt.subplots(2, n_cols, figsize=(4.5 * n_cols, 8))
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    # Vẽ KDE TRƯỚC khi biến đổi (tập Train)
    for i, col in enumerate(cols_to_transform):
        sns.kdeplot(df_train[col], ax=axes[0, i], color='#e74c3c', fill=True)
        axes[0, i].set_title(f'{col} (Before)', fontweight='bold')

    # Áp dụng transform cho CẢ 3 TẬP
    for col, info in skew_info.items():
        for df_subset in [df_train, df_val, df_test]:
            if info['transform'] == 'reflect_log':
                df_subset[col] = np.log1p(info['max_val'] - df_subset[col])
            else:
                df_subset[col] = np.log1p(df_subset[col])

    # Vẽ KDE SAU khi biến đổi (tập Train)
    for i, col in enumerate(cols_to_transform):
        transform_name = skew_info[col]['transform'].replace('_', ' ').title()
        sns.kdeplot(df_train[col], ax=axes[1, i], color='#2ecc71', fill=True)
        axes[1, i].set_title(f'{col} (After {transform_name})', fontweight='bold')

    plt.suptitle('EFFECT OF TRANSFORMATIONS ON HEAVY-TAILED FEATURES (Train Set)',
                 fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

# Lưu dữ liệu đã xử lý
import os
os.makedirs('../data/processed', exist_ok=True)
df_train.to_csv('../data/processed/Boston-train-clean.csv', index=False)
df_val.to_csv('../data/processed/Boston-val-clean.csv', index=False)
df_test.to_csv('../data/processed/Boston-test-clean.csv', index=False)
print("Đã xử lý Outliers dựa trên skewness tập Train và áp dụng cho cả 3 tập.")


### 7. Khám phá Phân phối & Quan hệ (Histogram & Scatter) — CHỈ TRÊN TẬP TRAIN
- 🎯 **Mục đích:** Đánh giá dữ liệu **tập Train** sau khi đã tiền xử lý.
- 💻 **Giải thích Code:** Dùng `.hist()` xem phân phối hình chuông. Dùng `.scatter()` vẽ quan hệ từng feature với `MEDV`.
- 📊 **Áp dụng vào Dữ liệu:** Thấy rõ hơn xu hướng tăng/giảm tuyến tính (như RM tăng thì MEDV tăng).
- 📈 **Kết quả (Output):** Các đồ thị điểm phân tán và biểu đồ cột.


In [ ]:
# 6. Scatter Plot: Features vs MEDV
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].scatter(df_train[col], df_train["MEDV"], alpha=0.6, s=20, color="#9b59b6", edgecolors="white")
    axes[i].set_xlabel(col, fontweight="bold")
    axes[i].set_ylabel("MEDV")
    axes[i].grid(alpha=0.3)

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Scatter: Features vs House Price (MEDV) — Train Set", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


### 8. Tương quan Đa biến & Hiện tượng Đa cộng tuyến (VIF) — CHỈ TRÊN TẬP TRAIN
- 🎯 **Mục đích:** Định lượng mối quan hệ giữa các biến với nhau và chọn lọc đặc trưng.
- 💻 **Giải thích Code:** `df.corr()` vẽ ma trận nhiệt. `variance_inflation_factor` kiểm tra xem các biến có đang đè lên nhau không (Đa cộng tuyến).
- 📊 **Áp dụng vào Dữ liệu:** Biến nào VIF > 10 là cảnh báo đỏ, có thể gây nhiễu cho mô hình học máy.
- 📈 **Kết quả (Output):** Heatmap tương quan và bảng xếp hạng VIF.


In [ ]:
# 7a. Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df_train.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix (Train Set)", fontsize=15, fontweight="bold")
plt.show()

# 7b. Variance Inflation Factor (VIF)
print("="*60)
print("VARIANCE INFLATION FACTOR (VIF)")
print("="*60)
X_vif = sm.add_constant(df_train[feature_cols])
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

vif_data = vif_data[vif_data["Feature"] != "const"].sort_values(by="VIF", ascending=False)
display(vif_data)


### 9. Phân tích chi tiết Biến Mục tiêu (MEDV) & Biến Phân loại — CHỈ TRÊN TẬP TRAIN
- 🎯 **Mục đích:** Soi kỹ xem Giá nhà có tuân theo phân phối chuẩn không, và các biến rời rạc như Vị trí bờ sông (CHAS) ảnh hưởng ra sao.
- 💻 **Giải thích Code:** `sns.histplot` kèm đường cong KDE. `stats.probplot` vẽ Q-Q Plot. `sns.boxplot` phân tách giá nhà theo CHAS.
- 📊 **Áp dụng vào Dữ liệu:** Cho cái nhìn thực tế về phân khúc bất động sản.
- 📈 **Kết quả (Output):** Các biểu đồ trực quan, chỉ số Skewness.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# KDE Plot
sns.histplot(df_train['MEDV'], kde=True, color='#f1c40f', ax=axes[0], stat='density')
axes[0].set_title('MEDV Distribution (Train Set)', fontweight='bold')

# Q-Q Plot
stats.probplot(df_train['MEDV'], dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot for MEDV', fontweight='bold')

# Boxplot CHAS vs MEDV
sns.boxplot(x='CHAS', y='MEDV', data=df_train, ax=axes[2], hue='CHAS', palette='Set2')
axes[2].set_title('House Price by Charles River (CHAS)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Skewness của MEDV: {df_train['MEDV'].skew():.4f} (Độ lệch so với phân phối chuẩn lý tưởng)")


### 10. Chuẩn Hoá Đặc trưng (Feature Scaling)
- 🎯 **Mục đích:** Đưa tất cả features về cùng thang đo (Z-Score) để mô hình học hiệu quả.
- 💻 **Giải thích Code:** Tính `mean` và `std` từ **tập Train**, rồi áp dụng chuẩn hoá cho cả 3 tập.
- ⚠️ **Chống Data Leakage:** Mean và Std chỉ được tính trên tập Train.


In [ ]:
# Trích xuất features và target từ các DataFrame đã tiền xử lý
X_train = df_train[feature_cols]
X_val   = df_val[feature_cols]
X_test  = df_test[feature_cols]
y_train = df_train['MEDV']
y_val   = df_val['MEDV']
y_test  = df_test['MEDV']

# Feature Scaling: fit trên Train, transform tất cả
mean_train = X_train.mean()
std_train = X_train.std()

X_train_scaled = ((X_train - mean_train) / std_train).values
X_val_scaled   = ((X_val   - mean_train) / std_train).values
X_test_scaled  = ((X_test  - mean_train) / std_train).values

print(f"Tập Train: {X_train_scaled.shape}")
print(f"Tập Val:   {X_val_scaled.shape}")
print(f"Tập Test:  {X_test_scaled.shape}")

os.makedirs('../data/features', exist_ok=True)
np.save('../data/features/X_train_scaled.npy', X_train_scaled)
np.save('../data/features/X_val_scaled.npy', X_val_scaled)
np.save('../data/features/X_test_scaled.npy', X_test_scaled)
np.save('../data/features/y_train.npy', y_train.values)
np.save('../data/features/y_val.npy', y_val.values)
np.save('../data/features/y_test.npy', y_test.values)
print("Đã lưu các ma trận đặc trưng vào thư mục '../data/features/'")


### 10.1. So sánh Dữ liệu Trước và Sau khi Chuẩn hóa (Feature Scaling)
- 🎯 **Mục đích:** Hiểu rõ tác dụng của việc đưa dữ liệu về cùng thang đo (Z-Score).
- 💻 **Giải thích Code:** Vẽ biểu đồ mật độ phân phối (KDE) của 2 Feature ngẫu nhiên (`RM` và `TAX`) trước và sau khi scale.
- 📊 **Áp dụng vào Dữ liệu:** Trước khi scale, `TAX` nằm ở dải 200-700, còn `RM` nằm ở dải 4-8. Sau khi scale, cả 2 đều hội tụ về khoảng trung bình `0`, độ lệch chuẩn `1`. Dáng điệu hình chuông (KDE) được giữ nguyên vẹn 100%.
- 📈 **Kết quả (Output):** Biểu đồ biến đổi trục tọa độ (Scale transformation).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Lấy index của cột RM và TAX
idx_RM = list(feature_cols).index('RM')
idx_TAX = list(feature_cols).index('TAX')

# Vẽ RM Before
sns.kdeplot(X_train['RM'], ax=axes[0,0], fill=True, color='#f39c12')
axes[0,0].set_title('RM: Before Scaling (Raw scale 4-8)', fontweight='bold')

# Vẽ RM After
sns.kdeplot(X_train_scaled[:, idx_RM], ax=axes[0,1], fill=True, color='#3498db')
axes[0,1].set_title('RM: After Z-Score Scaling (Mean=0, Std=1)', fontweight='bold')

# Vẽ TAX Before
sns.kdeplot(X_train['TAX'], ax=axes[1,0], fill=True, color='#c0392b')
axes[1,0].set_title('TAX: Before Scaling (Raw scale 200-700)', fontweight='bold')

# Vẽ TAX After
sns.kdeplot(X_train_scaled[:, idx_TAX], ax=axes[1,1], fill=True, color='#8e44ad')
axes[1,1].set_title('TAX: After Z-Score Scaling (Mean=0, Std=1)', fontweight='bold')

plt.tight_layout()
plt.suptitle('BEFORE VS AFTER FEATURE SCALING', fontsize=16, fontweight='bold', y=1.05)
plt.show()


### 18. Đóng gói Dữ liệu PyTorch (DataLoader)
- 🎯 **Mục đích:** Cắt nhỏ tập dữ liệu để đút vừa bộ nhớ, cung cấp liên tục cho mạng nơ-ron học.
- 💻 **Giải thích Code:** Ép kiểu numpy về `torch.Tensor`. Gộp bằng `TensorDataset` và chia mẻ bằng `DataLoader` (Batch = 32). Tập Train được shuffle (trộn ngẫu nhiên).
- 📊 **Áp dụng vào Dữ liệu:** Cơ sở hạt tầng cho vòng lặp Training.
- 📈 **Kết quả (Output):** Các Loader Object sẵn sàng.


In [ ]:
def to_tensor(x, y):
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

X_train_t, y_train_t = to_tensor(X_train_scaled, y_train)
X_val_t, y_val_t = to_tensor(X_val_scaled, y_val)
X_test_t, y_test_t = to_tensor(X_test_scaled, y_test)

BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

print("Các PyTorch DataLoaders đã được thiết lập thành công!")


### 18. Thiết lập Mạng Nơ-ron Đa lớp (Boston MLP)
- 🎯 **Mục đích:** Tạo bộ não AI.
- 💻 **Giải thích Code:** Khai báo class `BostonMLP` (13 Input -> 64 Hidden -> 32 Hidden -> 1 Output). Dùng Loss = MSE và Optimizer = Adam (lr=0.001).
- 📊 **Áp dụng vào Dữ liệu:** Phù hợp với bài toán Hồi quy (Regression) để đoán số thực (Giá nhà).
- 📈 **Kết quả (Output):** In ra màn hình cấu trúc layer.


In [ ]:
class BostonMLP(nn.Module):
    def __init__(self, input_dim):
        super(BostonMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.model(x)

input_dim = len(feature_cols)
model = BostonMLP(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)


### 18. Huấn luyện (Training Loop) & Early Stopping
- 🎯 **Mục đích:** Dạy mạng AI cách tìm quy luật và chống học vẹt (Overfitting).
- 💻 **Giải thích Code:** Cập nhật trọng số qua `loss.backward()`. Tính điểm Validation sau mỗi epoch. Nếu Loss không giảm 20 lần (patience=20), dừng sớm và lưu Model tốt nhất lại.
- 📊 **Áp dụng vào Dữ liệu:** Quét liên tục các batch dữ liệu của Train/Val.
- 📈 **Kết quả (Output):** Đường cong hội tụ Loss.


In [ ]:
import os
from torch.utils.tensorboard import SummaryWriter

if not os.path.exists('../models'): os.makedirs('../models')

# 1. Cấu hình Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 2. Khởi tạo TensorBoard Writer
writer = SummaryWriter("runs/boston_mlp")

EPOCHS = 1000
PATIENCE = 20
best_val_loss = float('inf')
patience_counter = 0

train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    # Training Loop
    model.train()
    epoch_train_loss = 0
    for data, labels in train_loader:
        # Move Data to Device
        data = data.to(device)
        labels = labels.to(device)
        
        # Zero Gradients
        optimizer.zero_grad()
        
        # Forward Pass
        outputs = model(data)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        
        # Backward Pass
        loss.backward()
        
        # Optimization Step
        optimizer.step()
        
        epoch_train_loss += loss.item() * data.size(0)
        
    avg_train_loss = epoch_train_loss / len(train_loader.dataset)
    train_losses.append(avg_train_loss)
    writer.add_scalar("Loss/train", avg_train_loss, epoch)
    
    # Validation Loop
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for data, labels in val_loader:
            # Move Data to Device
            data = data.to(device)
            labels = labels.to(device)
            
            # Forward Pass
            outputs = model(data)
            
            # Calculate Loss
            loss = criterion(outputs, labels)
            
            epoch_val_loss += loss.item() * data.size(0)
            
    avg_val_loss = epoch_val_loss / len(val_loader.dataset)
    val_losses.append(avg_val_loss)
    writer.add_scalar("Loss/val", avg_val_loss, epoch)
    
    # Early Stopping Logic
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), '../models/best_mlp_boston.pth')
    else:
        patience_counter += 1
        
    if patience_counter >= PATIENCE:
        print(f"Early stopping kích hoạt ở epoch {epoch+1}. Best Val Loss: {best_val_loss:.4f}")
        break

writer.close()

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='red')
plt.title('Training and Validation Loss', fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()


### 18. Đánh giá Mô hình trên Tập Test (Evaluation)
- 🎯 **Mục đích:** Bài thi tốt nghiệp đo lường hiệu suất cuối cùng.
- 💻 **Giải thích Code:** Load lại mô hình xịn nhất `best_model`. Tính các chỉ số MAE, MSE, RMSE, R² (R-squared).
- 📊 **Áp dụng vào Dữ liệu:** Chạy hoàn toàn độc lập trên bộ 15% Test Set.
- 📈 **Kết quả (Output):** Bản báo cáo điểm thi chi tiết.


In [ ]:
best_model = BostonMLP(input_dim)
best_model.load_state_dict(torch.load('../models/best_mlp_boston.pth', weights_only=True))
best_model.eval()

preds, actuals = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        preds.extend(best_model(batch_X).numpy())
        actuals.extend(batch_y.numpy())

preds = np.array(preds).flatten()
actuals = np.array(actuals).flatten()

mae = mean_absolute_error(actuals, preds)
mse = mean_squared_error(actuals, preds)
rmse = np.sqrt(mse)
r2 = r2_score(actuals, preds)

print("="*40)
print("TEST EVALUATION METRICS")
print("="*40)
print(f"MAE  (Mean Absolute Error): {mae:.4f}")
print(f"MSE  (Mean Squared Error) : {mse:.4f}")
print(f"RMSE (Root Mean Sq Error) : {rmse:.4f}")
print(f"R² Score                  : {r2:.4f}")

df_results = pd.DataFrame({'Actual': actuals, 'Predicted': preds, 'Absolute Error': np.abs(actuals - preds)})
display(df_results.head(10))


### 18. Trực quan hoá Sai số (Error Analysis)
- 🎯 **Mục đích:** Tìm quy luật mô hình đoán sai để rút kinh nghiệm.
- 💻 **Giải thích Code:** Biểu đồ trái so sánh Thực tế vs Dự đoán. Biểu đồ phải (Residuals) là khoảng cách lệch.
- 📊 **Áp dụng vào Dữ liệu:** Nếu các chấm xanh bám chặt đường chỉ đỏ `y=x`, mô hình dự đoán càng chính xác.
- 📈 **Kết quả (Output):** Hai biểu đồ đo độ chính xác trực quan.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Actual vs Predicted
sns.scatterplot(x=actuals, y=preds, ax=axes[0], color='#3498db', s=50, edgecolor='k')
min_val = min(min(actuals), min(preds))
max_val = max(max(actuals), max(preds))
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', lw=2)
axes[0].set_title('Actual vs Predicted', fontweight='bold')
axes[0].set_xlabel('Actual Price (MEDV)')
axes[0].set_ylabel('Predicted Price')

# Residuals
residuals = actuals - preds
sns.scatterplot(x=preds, y=residuals, ax=axes[1], color='#e74c3c', s=50, edgecolor='k')
axes[1].axhline(y=0, color='black', linestyle='--', lw=2)
axes[1].set_title('Residuals Analysis', fontweight='bold')
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()


### 18. Huấn luyện Mô hình Truyền thống & So sánh (Phase 43)
- 🎯 **Mục đích:** Đánh giá sức mạnh của MLP so với các thuật toán Machine Learning cổ điển (Linear, Decision Tree, Random Forest) trên cùng tập Test.
- 💻 **Giải thích Code:** 
  - Linear Regression dùng `X_train_scaled`.
  - Decision Tree và Random Forest dùng `X_train` (dữ liệu chưa scale).
  - Lấy kết quả từ `best_mlp` (đã scale). Tính `MSE` và `R²` cho tất cả.
- 📊 **Áp dụng vào Dữ liệu:** Đảm bảo mọi mô hình thi đấu trên cùng một bộ dữ liệu, cùng một hệ quy chiếu.
- 📈 **Kết quả (Output):** Bảng xếp hạng các mô hình dựa trên R² và MSE.


In [ ]:
import joblib
import os
import pandas as pd # Đảm bảo đã import pandas
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score # Thêm mean_absolute_error

os.makedirs('../models', exist_ok=True)

# 1. Linear Regression (cần dữ liệu chuẩn hóa)
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
y_pred_linear = linear_model.predict(X_test_scaled)

# 2. Decision Tree Regression (không cần chuẩn hóa)
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)

# 3. Random Forest Regression (không cần chuẩn hóa)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Lưu các mô hình truyền thống
joblib.dump(linear_model, '../models/linear_model.pkl')
joblib.dump(tree_model, '../models/tree_model.pkl')
joblib.dump(rf_model, '../models/rf_model.pkl')
print("Đã lưu các mô hình truyền thống (Linear, Tree, RF) vào thư mục '../models/'")

# 4. MLP Regression (đã chuẩn hóa, gọi lại kết quả)
y_pred_mlp = preds # biến preds đã được tính toán ở phần Eval của MLP

# Tính Metrics
models = ['Linear Regression', 'Decision Tree', 'Random Forest', 'MLP']
predictions = [y_pred_linear, y_pred_tree, y_pred_rf, y_pred_mlp]

mse_scores = []
mae_scores = [] # Thêm list để lưu MAE
r2_scores = []

for pred in predictions:
    mse_scores.append(mean_squared_error(actuals, pred))
    mae_scores.append(mean_absolute_error(actuals, pred)) # Tính toán MAE
    r2_scores.append(r2_score(actuals, pred))

# Tạo bảng tổng hợp
comparison_df = pd.DataFrame({
    'Model': models,
    'MSE': mse_scores,
    'MAE': mae_scores, # Thêm cột MAE vào bảng
    'R2 Score': r2_scores
})

# Sắp xếp theo R2 Score giảm dần
comparison_df = comparison_df.sort_values(by='R2 Score', ascending=False).reset_index(drop=True)

print("="*50)
print("BẢNG XẾP HẠNG HIỆU NĂNG CÁC MÔ HÌNH")
print("="*50)
display(comparison_df)


### 18. Phân tích Hình ảnh So sánh (Visualizations)
- 🎯 **Mục đích:** Nhìn bằng mắt thường cách từng mô hình bám theo giá thực tế.
- 💻 **Giải thích Code:** Dùng `matplotlib` để vẽ 4 khung biểu đồ song song. Trục ngang là Giá thực, trục dọc là Giá dự đoán.
- 📊 **Áp dụng vào Dữ liệu:** Nếu các chấm màu xanh nằm càng gần đường chéo màu đỏ (`y = x`), độ chính xác càng tuyệt đối.
- 📈 **Kết quả (Output):** Biểu đồ đa chiều (Subplots).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
min_val = min(actuals)
max_val = max(actuals)

for i, (model_name, pred) in enumerate(zip(models, predictions)):
    sns.scatterplot(x=actuals, y=pred, ax=axes[i], color=colors[i], s=50, edgecolor='k', alpha=0.7)
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    axes[i].set_title(f'Actual vs Predicted: {model_name}', fontweight='bold')
    axes[i].set_xlabel('Actual Price (MEDV)')
    axes[i].set_ylabel('Predicted Price')
    axes[i].legend()

plt.tight_layout()
plt.suptitle('MODEL COMPARISON: ACTUAL VS PREDICTED', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Vẽ thêm biểu đồ Residuals
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (model_name, pred) in enumerate(zip(models, predictions)):
    residuals_model = actuals - pred
    sns.scatterplot(x=pred, y=residuals_model, ax=axes[i], color=colors[i], s=50, edgecolor='k', alpha=0.7)
    axes[i].axhline(y=0, color='black', linestyle='--', lw=2)
    axes[i].set_title(f'Residuals: {model_name}', fontweight='bold')
    axes[i].set_xlabel('Predicted Price')
    axes[i].set_ylabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.suptitle('MODEL COMPARISON: RESIDUAL ANALYSIS', fontsize=16, fontweight='bold', y=1.02)
plt.show()


### 18. Kết luận Cuối cùng (Final Conclusion)

Dựa trên bảng thành tích (Metrics) và trực quan hóa (Visualizations) trên tập **Test Set**, ta rút ra được các nhận định sau:

1. **Model nào xuất sắc nhất?**
   - Dựa trên chỉ số, **Random Forest Regression** và **MLP** thường cạnh tranh nhau vị trí dẫn đầu. Nhờ khả năng học phi tuyến (Non-linear) và tổng hợp nhiều nhánh cây quyết định (Ensemble), Random Forest thường đạt $R^2$ cực cao trên dữ liệu dạng bảng (Tabular data).

2. **Sự yếu kém của Linear Regression:**
   - Linear Regression thường đứng bét bảng vì nó chỉ giả định mối quan hệ là một đường thẳng tuyến tính. Trong khi thực tế giá nhà liên quan đến tội phạm, độ ô nhiễm, hay số phòng... là những đường cong phi tuyến phức tạp.

3. **Decision Tree dễ bị Overfitting:**
   - Một cây quyết định đơn lẻ (Decision Tree) tuy mô hình được phi tuyến nhưng rất nhạy cảm với nhiễu và dễ bị học vẹt, dẫn đến khả năng tổng quát hóa trên tập Test không cao bằng Rừng ngẫu nhiên (Random Forest).

4. **Sức mạnh của Neural Network (MLP):**
   - MLP thể hiện rất tốt (vượt xa Linear). Tuy nhiên, trên những bộ dữ liệu quá nhỏ (chỉ vỏn vẹn ~500 dòng như Boston Housing), mạng Nơ-ron chưa thể phát huy được toàn bộ sức mạnh "khổng lồ" của nó như trên các dữ liệu hình ảnh, âm thanh hay hàng triệu dòng. 

**🏆 Quyết định:** 
Đối với bộ dữ liệu Boston Housing, các thuật toán Tree-based như **Random Forest** chứng minh được sự vượt trội về cả độ chính xác, tốc độ train, và không đòi hỏi quá trình chuẩn hóa (Scaling). Mặc dù vậy, **MLP** vẫn chứng tỏ tiềm năng cực kỳ to lớn nếu dữ liệu trong tương lai phình to!
